## Part 1: Explore the Data

Import the data and use Pandas to learn more about the dataset.

In [9]:
import pandas as pd
import numpy as np

df = pd.read_csv('Resources/client_dataset.csv')
df.head()

,first,last,job,phone,email,client_id,order_id,order_date,order_week,order_year,item_id,category,subcategory,unit_price,unit_cost,unit_weight,qty,line_number
0,Donald,Harding,Immunologist,793-904-7725x39308,harding.donald.7185@sullivan.com,58515,8953482,2023-04-28,17,2023,EUD29711-63-6U,decor,wall art,1096.80,762.71,7.50,105,1
1,Tiffany,Myers,Music therapist,201.442.4543x942,myers.t.6537@ferguson-johnson.net,37609,8069089,2023-05-19,20,2023,XDA18116-89-4A,consumables,pens,24.95,15.09,1.49,21,0
2,Shannon,Watson,Immunologist,687.737.9424x8503,swatson8146@payne.net,57113,1902144,2023-01-29,4,2023,ABE59463-05-7E,software,project management,13.52,7.86,1.68,39,6
3,Nathan,Baker,Accounting technician,827-788-8123x012,bakernathan@benson.com,46554,9031802,2023-04-25,17,2023,ZMM00836-65-0C,consumables,pens,36.42,24.85,1.23,29,3
4,Christina,Schwartz,Chiropractor,265-829-3643,christinaschwartz9252@mcconnell.com,92089,1322274,2023-05-28,21,2023,BZX55559-12-3X,consumables,misc,195.10,108.17,46.43,20,1


In [10]:
# View the column names in the data
df.columns

Index(['first', 'last', 'job', 'phone', 'email', 'client_id', 'order_id',
       'order_date', 'order_week', 'order_year', 'item_id', 'category',
       'subcategory', 'unit_price', 'unit_cost', 'unit_weight', 'qty',
       'line_number'],
      dtype='object')

In [11]:
# Use the describe function to gather some basic statistics
describe = df.describe()
print(describe)

          client_id      order_id    order_week    order_year    unit_price  \
count  54639.000000  5.463900e+04  54639.000000  54639.000000  54639.000000   
mean   54837.869416  5.470190e+06     11.359139   2022.993064    136.267207   
std    25487.438231  2.599807e+06      7.023499      0.082997    183.873135   
min    10033.000000  1.000886e+06      1.000000   2022.000000      0.010000   
25%    33593.000000  3.196372e+06      6.000000   2023.000000     20.800000   
50%    53305.000000  5.496966e+06     11.000000   2023.000000     68.310000   
75%    78498.000000  7.733869e+06     17.000000   2023.000000    173.160000   
max    99984.000000  9.998480e+06     52.000000   2023.000000   1396.230000   

          unit_cost   unit_weight           qty   line_number  
count  54639.000000  54639.000000  5.463900e+04  54639.000000  
mean      99.446073      5.004116  5.702646e+02      2.979667  
std      133.164267      5.326599  1.879552e+04      2.436320  
min        0.010000      0.00000

In [12]:
# Use this space to do any additional research
# and familiarize yourself with the data.



In [17]:
# What three item categories had the most entries?
most_entries = df['category'].value_counts().head(3)
print(most_entries)

category
consumables    23538
furniture      11915
software        8400
Name: count, dtype: int64


The top three categories are consumables, furniture, and software respectively. 

In [19]:
# For the category with the most entries, which subcategory had the most entries?

# Find the category with the most entries
top_category = df['category'].value_counts().idxmax()

# Filter the DataFrame for the top category
top_category_df = df[df['category'] == top_category]

# Find the subcategory with the most entries within the top category
top_subcategory = top_category_df['subcategory'].value_counts().idxmax()
top_subcategory_count = top_category_df['subcategory'].value_counts().max()

print(f"The subcategory with the most entries in the top category '{top_category}' is '{top_subcategory}' with {top_subcategory_count} entries.")

The subcategory with the most entries in the top category 'consumables' is 'bathroom supplies' with 6424 entries.


In [20]:
# Which five clients had the most entries in the data?
top_clients = df['client_id'].value_counts().head(5)
print(top_clients)

client_id
33615    220
66037    211
46820    209
38378    207
24741    207
Name: count, dtype: int64


In [21]:
# Store the client ids of those top 5 clients in a list.
top_clients_list = df['client_id'].value_counts().head(5).index.tolist()
print(top_clients_list)

[33615, 66037, 46820, 38378, 24741]


In [24]:

# Find the client with the most entries
top_client_id = df['client_id'].value_counts().idxmax()

# Calculate the total units ordered by this client
total_units_ordered = df[df['client_id'] == top_client_id]['qty'].sum()

print(f"The client with the most entries (ID: {top_client_id}) ordered a total of {total_units_ordered} units.")

The client with the most entries (ID: 33615) ordered a total of 64313 units.


## Part 2: Transform the Data
Do we know that this client spent the more money than client 66037? If not, how would we find out? Transform the data using the steps below to prepare it for analysis.

In [25]:
# Create a column that calculates the subtotal for each line using the unit_price and the qty
df['subtotal'] = df['unit_price'] * df['qty']

In [29]:
# Create a column for shipping price.
# Assume a shipping price of $7 per pound for orders over 50 pounds and $10 per pound for items 50 pounds or under.
df['shipping_price'] = df.apply(lambda row: row['unit_weight'] * row['qty'] * (7 if row['unit_weight'] * row['qty'] > 50 else 10), axis=1)



In [30]:
# Create a column for the total price using the subtotal and the shipping price along with a sales tax of 9.25%
df['total_price'] = (df['subtotal'] + df['shipping_price']) * 1.0925

In [31]:
# Create a column for the cost of each line using unit cost, qty, and
# shipping price (assume the shipping cost is exactly what is charged to the client).
tax_rate = 0.0925
df['total_price'] = (df['subtotal'] + df['shipping_price']) * (1 + tax_rate)


In [32]:
# Create a column for the profit of each line using line cost and line price
df['line_cost'] = (df['unit_cost'] * df['qty']) + df['shipping_price']


## Part 3: Confirm your work
You have email receipts showing that the total prices for 3 orders. Confirm that your calculations match the receipts. Remember, each order has multiple lines.

Order ID 2742071 had a total price of \$152,811.89

Order ID 2173913 had a total price of \$162,388.71

Order ID 6128929 had a total price of \$923,441.25


In [33]:
# Check your work using the totals above

# List of order IDs to check
order_ids = [2742071, 2173913, 6128929]

# Check the total price for each order ID
for order_id in order_ids:
    total_price = df[df['order_id'] == order_id]['total_price'].sum()
    print(f"Order ID {order_id} has a calculated total price of ${total_price:.2f}")

Order ID 2742071 has a calculated total price of $152811.90
Order ID 2173913 has a calculated total price of $162388.72
Order ID 6128929 has a calculated total price of $923441.24


## Part 4: Summarize and Analyze
Use the new columns with confirmed values to find the following information.

In [34]:
# How much did each of the top 5 clients by quantity spend? Check your work from Part 1 for client ids.

# List of order IDs to check
order_ids = [2742071, 2173913, 6128929]

# Verify the orders
order_verification = df[df['order_id'].isin(order_ids)][['order_id', 'subtotal', 'shipping_price', 'total_price', 'line_cost']]
print("\nOrder Verification:")
print(order_verification)

# Calculate the total amount spent by each of the top 5 clients by quantity
top_clients_list = df['client_id'].value_counts().head(5).index.tolist()
client_spending = df[df['client_id'].isin(top_clients_list)].groupby('client_id')['total_price'].sum()

print("\nTop 5 Clients by Quantity and Their Total Spending:")
print(client_spending)


Order Verification:
       order_id   subtotal  shipping_price    total_price  line_cost
627     2173913    7718.90         1303.40    9856.862750    8039.50
3422    6128929    8890.75        35744.17   48763.650100   43628.42
3446    2742071     430.43         5637.94    6629.694225    5872.79
6287    2742071  119652.00         4732.00  135889.520000  110728.80
13672   6128929    5459.60         1974.00    8121.208000    5578.80
14089   2173913     118.08          374.40     538.034400     449.64
14517   6128929  117187.29        47872.44  180327.755025  154975.59
14938   2173913    5835.33          656.88    7092.739425    4348.38
14962   2173913    2145.78         2500.47    5076.028125    3635.73
16295   6128929    3439.92          343.20    4133.058600    2222.88
20623   2742071    1621.44          272.40    2069.020200    1241.76
21141   6128929     731.12          837.20    1713.389600    1451.84
22004   2173913    8655.57          168.30    9640.077975    4686.66
23840   27420

In [36]:
# Create a summary DataFrame showing the totals for the for the top 5 clients with the following information:
# total units purchased, total shipping price, total revenue, and total profit. 

# Calculate the profit for each line
df['profit'] = df['subtotal'] - df['line_cost']

# List of top 5 client IDs
top_5_clients = [33615, 66037, 46820, 38378, 24741]

# Filter the DataFrame for the top 5 clients
top_clients_data = df[df['client_id'].isin(top_5_clients)]

# Group by client_id and calculate the required summaries
top_5_summary = top_clients_data.groupby('client_id').agg(
    Total_Units=('qty', 'sum'),
    Total_Shipping=('shipping_price', 'sum'),
    Total_Revenue=('total_price', 'sum'),
    Total_Profit=('profit', 'sum')
).reset_index()

print(top_5_summary)


   client_id  Total_Units  Total_Shipping  Total_Revenue  Total_Profit
0      24741       239862      5126448.37   8.226889e+07   24487985.54
1      33615        64313      1828984.89   8.377309e+06    -336281.80
2      38378        73667      3429455.40   1.290655e+07   -1250399.83
3      46820        75768      1601448.84   9.743794e+06     310164.39
4      66037        43018      1395151.85   1.025951e+07     991225.40


In [38]:
# Format the data and rename the columns to names suitable for presentation.

# Ensure the profit column is created
df['profit'] = df['subtotal'] - df['line_cost']

# List of top 5 client IDs
top_5_clients = [33615, 66037, 46820, 38378, 24741]

# Filter the DataFrame for the top 5 clients
top_clients_data = df[df['client_id'].isin(top_5_clients)]

# Group by client_id and calculate the required summaries
top_5_summary = top_clients_data.groupby('client_id').agg(
    Total_Units_Purchased=('qty', 'sum'),
    Shipping_Cost_Millions=('shipping_price', lambda x: x.sum() / 1_000_000),
    Total_Revenue_Millions=('total_price', lambda x: x.sum() / 1_000_000),
    Total_Profit=('profit', 'sum')
).reset_index()

# Rename the columns to reflect the change in the money format
top_5_summary = top_5_summary.rename(columns={'client_id': 'Client ID'})

print(top_5_summary)

   Client ID  Total_Units_Purchased  Shipping_Cost_Millions  \
0      24741                 239862                5.126448   
1      33615                  64313                1.828985   
2      38378                  73667                3.429455   
3      46820                  75768                1.601449   
4      66037                  43018                1.395152   

   Total_Revenue_Millions  Total_Profit  
0               82.268892   24487985.54  
1                8.377309    -336281.80  
2               12.906551   -1250399.83  
3                9.743794     310164.39  
4               10.259515     991225.40  


In [40]:
# Sort the updated data by "Total Profit (millions)" form highest to lowest and assign the sort to a new DatFrame.
final_summary = top_5_summary.sort_values('Total_Profit', ascending=False)

print("\nFinal Summary for Top 5 Clients:")
print(final_summary)


Final Summary for Top 5 Clients:
   Client ID  Total_Units_Purchased  Shipping_Cost_Millions  \
0      24741                 239862                5.126448   
4      66037                  43018                1.395152   
3      46820                  75768                1.601449   
1      33615                  64313                1.828985   
2      38378                  73667                3.429455   

   Total_Revenue_Millions  Total_Profit  
0               82.268892   24487985.54  
4               10.259515     991225.40  
3                9.743794     310164.39  
1                8.377309    -336281.80  
2               12.906551   -1250399.83  
